In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

In [2]:
ROOT = Path.cwd().parent

RUNTIME_DIR = ROOT / "dataset" / "runtime"
CHECKPOINT_DIR = ROOT / "checkpoints"

INPUT_PATH = RUNTIME_DIR / "comments_aspects.csv"
MODEL_PATH = CHECKPOINT_DIR / "atae_lstm_best.pt"
OUTPUT_PATH = RUNTIME_DIR / "atae_predictions.csv"

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Project root:", ROOT)
print("Input aspect CSV:", INPUT_PATH)
print("ATAE-LSTM checkpoint:", MODEL_PATH)
print("Output predictions:", OUTPUT_PATH)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

Project root: c:\New folder\Projects\sentiment analysis
Input aspect CSV: c:\New folder\Projects\sentiment analysis\dataset\runtime\comments_aspects.csv
ATAE-LSTM checkpoint: c:\New folder\Projects\sentiment analysis\checkpoints\atae_lstm_best.pt
Output predictions: c:\New folder\Projects\sentiment analysis\dataset\runtime\atae_predictions.csv
Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
CUDA version: 12.8


In [3]:
required_paths = [
    INPUT_PATH,
    MODEL_PATH
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required file(s) missing:\n"
        + "\n".join(str(path) for path in missing_paths)
        + "\n\nRun these notebooks first:\n"
        "1. 02_train_atae_lstm.ipynb\n"
        "2. 03_extract_aspects.ipynb"
    )

print("All required files found.")

All required files found.


In [4]:
def tokenize(text):
    text = str(text).lower()
    return re.findall(
        r"[a-z0-9]+(?:'[a-z]+)?",
        text
    )

In [5]:
PAD_ID = 0
UNK_ID = 1


def encode_tokens(tokens, vocabulary, max_length):
    token_ids = [
        vocabulary.get(token, UNK_ID)
        for token in tokens[:max_length]
    ]

    padding_needed = max_length - len(token_ids)

    if padding_needed > 0:
        token_ids.extend([PAD_ID] * padding_needed)

    return token_ids

In [6]:
class ATAELSTM(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_classes,
        embedding_weights=None,
        dropout=0.30
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_ID
        )

        # At inference time, the trained weights loaded from the checkpoint
        # will overwrite any initial random embedding weights.
        if embedding_weights is not None:
            self.embedding.weight.data.copy_(
                embedding_weights
            )

        self.lstm = nn.LSTM(
            input_size=embedding_dim * 2,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        lstm_output_dim = hidden_dim * 2

        self.attention_projection = nn.Linear(
            lstm_output_dim + embedding_dim,
            lstm_output_dim
        )

        self.attention_score = nn.Linear(
            lstm_output_dim,
            1,
            bias=False
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            lstm_output_dim,
            num_classes
        )

    def forward(self, sentence_ids, aspect_ids):
        sentence_mask = sentence_ids.ne(PAD_ID)

        sentence_embeddings = self.embedding(
            sentence_ids
        )

        aspect_embeddings = self.embedding(
            aspect_ids
        )

        aspect_mask = aspect_ids.ne(PAD_ID).unsqueeze(-1)

        aspect_sum = (
            aspect_embeddings * aspect_mask
        ).sum(dim=1)

        aspect_length = aspect_mask.sum(
            dim=1
        ).clamp(min=1)

        aspect_vector = aspect_sum / aspect_length

        expanded_aspect = aspect_vector.unsqueeze(1).expand(
            -1,
            sentence_embeddings.size(1),
            -1
        )

        lstm_input = torch.cat(
            [sentence_embeddings, expanded_aspect],
            dim=-1
        )

        lstm_output, _ = self.lstm(lstm_input)

        attention_input = torch.cat(
            [lstm_output, expanded_aspect],
            dim=-1
        )

        attention_hidden = torch.tanh(
            self.attention_projection(attention_input)
        )

        attention_scores = self.attention_score(
            attention_hidden
        ).squeeze(-1)

        attention_scores = attention_scores.masked_fill(
            ~sentence_mask,
            -1e9
        )

        attention_weights = torch.softmax(
            attention_scores,
            dim=1
        )

        sentence_representation = torch.bmm(
            attention_weights.unsqueeze(1),
            lstm_output
        ).squeeze(1)

        sentence_representation = self.dropout(
            sentence_representation
        )

        logits = self.classifier(
            sentence_representation
        )

        return logits, attention_weights

In [7]:
checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

print("Checkpoint keys:")
print(checkpoint.keys())

vocab = checkpoint["vocab"]
label_to_id = checkpoint["label_to_id"]
config = checkpoint["config"]

id_to_label = {
    label_id: label
    for label, label_id in label_to_id.items()
}

print("\nVocabulary size:", len(vocab))
print("Label mapping:", label_to_id)
print("Configuration:", config)

if "best_validation_macro_f1" in checkpoint:
    print(
        "Best validation Macro-F1:",
        round(
            checkpoint["best_validation_macro_f1"],
            4
        )
    )

Checkpoint keys:
dict_keys(['model_state_dict', 'vocab', 'label_to_id', 'config', 'best_validation_macro_f1'])

Vocabulary size: 1883
Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2, 'conflict': 3}
Configuration: {'embedding_dim': 100, 'hidden_dim': 128, 'dropout': 0.3, 'max_sentence_len': 100, 'max_aspect_len': 10}
Best validation Macro-F1: 0.4842


In [8]:
model = ATAELSTM(
    vocab_size=len(vocab),
    embedding_dim=config["embedding_dim"],
    hidden_dim=config["hidden_dim"],
    num_classes=len(label_to_id),
    dropout=config["dropout"]
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

MAX_SENTENCE_LEN = config["max_sentence_len"]
MAX_ASPECT_LEN = config["max_aspect_len"]

print("Model loaded successfully.")

Model loaded successfully.


In [9]:
aspect_data = pd.read_csv(INPUT_PATH)

print("Original aspect-row count:", len(aspect_data))
print("Columns:", aspect_data.columns.tolist())

required_columns = {
    "comment_id",
    "text",
    "aspect_term"
}

missing_columns = required_columns - set(aspect_data.columns)

if missing_columns:
    raise ValueError(
        "comments_aspects.csv is missing columns: "
        + ", ".join(sorted(missing_columns))
    )

aspect_data = aspect_data.dropna(
    subset=["comment_id", "text", "aspect_term"]
).copy()

aspect_data["text"] = (
    aspect_data["text"]
    .astype(str)
    .str.strip()
)

aspect_data["aspect_term"] = (
    aspect_data["aspect_term"]
    .astype(str)
    .str.strip()
    .str.lower()
)

aspect_data = aspect_data[
    aspect_data["text"].ne("") &
    aspect_data["aspect_term"].ne("")
].copy()

aspect_data = aspect_data.reset_index(drop=True)

print("Usable aspect-row count:", len(aspect_data))

display(aspect_data.head(10))

Original aspect-row count: 11
Columns: ['comment_id', 'text', 'raw_aspect', 'aspect_term']
Usable aspect-row count: 11


,comment_id,text,raw_aspect,aspect_term
0,c001,The laptop is excellent for school work and br...,8gb ram,8gb ram
1,c001,The laptop is excellent for school work and br...,school,school
2,c001,The laptop is excellent for school work and br...,browsing,productivity
3,c002,"Light games and older titles run fine, but AAA...",aaa,gaming performance
4,c003,"The touchscreen is useful, and Microsoft Offic...",office,office work
5,c003,"The touchscreen is useful, and Microsoft Offic...",microsoft office,productivity
6,c003,"The touchscreen is useful, and Microsoft Offic...",touchscreen,touchscreen
7,c004,"The laptop has poor gaming performance, althou...",gaming performance,gaming performance
8,c004,"The laptop has poor gaming performance, althou...",productivity,productivity
9,c005,"The battery life is decent, but storage fills ...",storage,storage


In [10]:
def predict_atae_polarity(
    text,
    aspect_term,
    model,
    vocabulary
):
    """
    Input:
        Raw comment text and one normalized aspect term.

    Output:
        predicted polarity, probability confidence,
        all class probabilities, and attention scores.
    """
    model.eval()

    text_tokens = tokenize(text)
    aspect_tokens = tokenize(aspect_term)

    sentence_ids = encode_tokens(
        tokens=text_tokens,
        vocabulary=vocabulary,
        max_length=MAX_SENTENCE_LEN
    )

    aspect_ids = encode_tokens(
        tokens=aspect_tokens,
        vocabulary=vocabulary,
        max_length=MAX_ASPECT_LEN
    )

    sentence_tensor = torch.tensor(
        [sentence_ids],
        dtype=torch.long
    ).to(device)

    aspect_tensor = torch.tensor(
        [aspect_ids],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():
        logits, attention_weights = model(
            sentence_tensor,
            aspect_tensor
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[0]

        predicted_label_id = torch.argmax(
            probabilities
        ).item()

    class_probabilities = {
        id_to_label[label_id]: round(
            float(probabilities[label_id].item()),
            4
        )
        for label_id in range(len(label_to_id))
    }

    return {
        "predicted_polarity": id_to_label[
            predicted_label_id
        ],
        "confidence": round(
            float(probabilities[predicted_label_id].item()),
            4
        ),
        "all_class_scores": class_probabilities,
        "attention_weights": attention_weights[
            0
        ].detach().cpu().numpy()
    }

In [14]:
sample_comment = """
The laptop works very well for school, browsing,
coding, and Microsoft Office. However, 8GB RAM is
restrictive, and AAA gaming will struggle.
"""

sample_aspect = "8gb ram"

sample_result = predict_atae_polarity(
    text=sample_comment,
    aspect_term=sample_aspect,
    model=model,
    vocabulary=vocab
)

print("Aspect:", sample_aspect)
print("Prediction:", sample_result["predicted_polarity"])
print("Confidence:", sample_result["confidence"])
print("All class scores:")
print(sample_result["all_class_scores"])

Aspect: 8gb ram
Prediction: negative
Confidence: 0.9506
All class scores:
{'negative': 0.9506, 'neutral': 0.0351, 'positive': 0.0096, 'conflict': 0.0047}


In [15]:
prediction_rows = []

for row_index, row in aspect_data.iterrows():
    prediction = predict_atae_polarity(
        text=row["text"],
        aspect_term=row["aspect_term"],
        model=model,
        vocabulary=vocab
    )

    prediction_rows.append(
        {
            "comment_id": row["comment_id"],
            "text": row["text"],
            "raw_aspect": (
                row["raw_aspect"]
                if "raw_aspect" in aspect_data.columns
                else None
            ),
            "aspect_term": row["aspect_term"],
            "predicted_polarity": prediction[
                "predicted_polarity"
            ],
            "confidence": prediction["confidence"],
            "all_class_scores": prediction[
                "all_class_scores"
            ],
            "model": "atae_lstm"
        }
    )

    if (row_index + 1) % 50 == 0:
        print(
            f"Processed {row_index + 1} "
            f"of {len(aspect_data)} rows"
        )

atae_predictions = pd.DataFrame(prediction_rows)

print("Total ATAE-LSTM predictions:", len(atae_predictions))

display(atae_predictions.head(20))

Total ATAE-LSTM predictions: 11


,comment_id,text,raw_aspect,aspect_term,predicted_polarity,confidence,all_class_scores,model
0,c001,The laptop is excellent for school work and br...,8gb ram,8gb ram,negative,0.6688,"{'negative': 0.6688, 'neutral': 0.109, 'positi...",atae_lstm
1,c001,The laptop is excellent for school work and br...,school,school,positive,0.7729,"{'negative': 0.1872, 'neutral': 0.0383, 'posit...",atae_lstm
2,c001,The laptop is excellent for school work and br...,browsing,productivity,positive,0.7052,"{'negative': 0.0609, 'neutral': 0.2326, 'posit...",atae_lstm
3,c002,"Light games and older titles run fine, but AAA...",aaa,gaming performance,negative,0.9803,"{'negative': 0.9803, 'neutral': 0.0048, 'posit...",atae_lstm
4,c003,"The touchscreen is useful, and Microsoft Offic...",office,office work,positive,0.6671,"{'negative': 0.2697, 'neutral': 0.0596, 'posit...",atae_lstm
5,c003,"The touchscreen is useful, and Microsoft Offic...",microsoft office,productivity,positive,0.5065,"{'negative': 0.0548, 'neutral': 0.4365, 'posit...",atae_lstm
6,c003,"The touchscreen is useful, and Microsoft Offic...",touchscreen,touchscreen,negative,0.4912,"{'negative': 0.4912, 'neutral': 0.4421, 'posit...",atae_lstm
7,c004,"The laptop has poor gaming performance, althou...",gaming performance,gaming performance,negative,0.9609,"{'negative': 0.9609, 'neutral': 0.0113, 'posit...",atae_lstm
8,c004,"The laptop has poor gaming performance, althou...",productivity,productivity,positive,0.7179,"{'negative': 0.1364, 'neutral': 0.1428, 'posit...",atae_lstm
9,c005,"The battery life is decent, but storage fills ...",storage,storage,negative,0.4351,"{'negative': 0.4351, 'neutral': 0.0804, 'posit...",atae_lstm


In [16]:
atae_predictions.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved ATAE-LSTM predictions to:")
print(OUTPUT_PATH)

Saved ATAE-LSTM predictions to:
c:\New folder\Projects\sentiment analysis\dataset\runtime\atae_predictions.csv


In [17]:
prediction_distribution = (
    atae_predictions
    .groupby(
        ["aspect_term", "predicted_polarity"]
    )
    .size()
    .reset_index(name="mentions")
    .sort_values(
        ["aspect_term", "mentions"],
        ascending=[True, False]
    )
)

display(prediction_distribution)

,aspect_term,predicted_polarity,mentions
0,8gb ram,negative,1
1,battery,negative,1
2,gaming performance,negative,2
3,office work,positive,1
4,productivity,positive,3
5,school,positive,1
6,storage,negative,1
7,touchscreen,negative,1


In [18]:
low_confidence_predictions = (
    atae_predictions
    .sort_values(
        "confidence",
        ascending=True
    )
    .head(20)
)

display(
    low_confidence_predictions[
        [
            "comment_id",
            "text",
            "aspect_term",
            "predicted_polarity",
            "confidence"
        ]
    ]
)

,comment_id,text,aspect_term,predicted_polarity,confidence
9,c005,"The battery life is decent, but storage fills ...",storage,negative,0.4351
6,c003,"The touchscreen is useful, and Microsoft Offic...",touchscreen,negative,0.4912
5,c003,"The touchscreen is useful, and Microsoft Offic...",productivity,positive,0.5065
10,c005,"The battery life is decent, but storage fills ...",battery,negative,0.5114
4,c003,"The touchscreen is useful, and Microsoft Offic...",office work,positive,0.6671
0,c001,The laptop is excellent for school work and br...,8gb ram,negative,0.6688
2,c001,The laptop is excellent for school work and br...,productivity,positive,0.7052
8,c004,"The laptop has poor gaming performance, althou...",productivity,positive,0.7179
1,c001,The laptop is excellent for school work and br...,school,positive,0.7729
7,c004,"The laptop has poor gaming performance, althou...",gaming performance,negative,0.9609
